In [0]:
client_id = dbutils.secrets.get(scope = "productsetl", key="client_id")
client_secret = dbutils.secrets.get(scope = "productsetl", key="client_secret")

In [0]:
import requests
import base64

def get_kroger_token(client_id, client_secret):
    credentials = base64.b64encode(f"{client_id}:{client_secret}".encode()).decode()
    response = requests.post(
        "https://api.kroger.com/v1/connect/oauth2/token",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Authorization": f"Basic {credentials}",
            "Accept": "application/json"
        },
        data={
            "grant_type": "client_credentials",
            "scope": "product.compact"
        }
    )
    response.raise_for_status()
    return response.json()["access_token"]

token = get_kroger_token(client_id, client_secret)

In [0]:
def fetch_products(token, search_term, location_id="01400943", limit=50):
    all_products = []
    start = 1
    while True:
        resp = requests.get(
            "https://api.kroger.com/v1/products",
            headers={"Authorization": f"Bearer {token}", "Accept": "application/json"},
            params={
                "filter.term": search_term,
                "filter.locationId": location_id,
                "filter.limit": limit,
                "filter.start": start
            }
        )
        resp.raise_for_status()
        data = resp.json().get("data", [])
        if not data:
            break
        all_products.extend(data)
        start += limit
        if start > 250:
            break
    return all_products

In [0]:
essential_grocery_terms = [
    "chicken",      # meat
    "salmon",       # seafood
    "rice",         # grains
    "milk",         # dairy
    "apple",        # fruit
    "carrot",       # vegetable
    "lentil",       # legume
    "basil",        # spice/herb
    "mayonnaise",   # condiment
    "chips"         # snack
]

In [0]:
# Fetch top 20 relevant products per term
products_by_term = {}
for term in essential_grocery_terms:
    results = fetch_products(token, search_term=term, limit=20)
    products_by_term[term] = results
    print(f"{term}: {len(results)} products fetched")

# Merge and deduplicate across all terms
all_products = []
for plist in products_by_term.values():
    all_products.extend(plist)
unique_products = {}
for p in all_products:
    pid = p.get("productId")
    upc = p.get("upc")
    key = (pid, upc)
    if key not in unique_products:
        unique_products[key] = p
products = list(unique_products.values())

print(f"Total unique products loaded: {len(products)}")


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS kroger")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
import datetime
import json

spark = SparkSession.builder.getOrCreate()

def flatten_products(products):
    now = datetime.datetime.now()
    rows = []
    for p in products:
        images = p.get("images", [{}])
        items = p.get("items", [{}])
        price = items[0].get("price", {}) if items else {}
        national_price = items[0].get("nationalPrice", {}) if items else {}
        regular = price.get("regular")
        promo = price.get("promo")
        national_regular = national_price.get("regular")
        national_promo = national_price.get("promo")
        # Compute price_of_day: promo price if available, else regular
        if promo is not None:
            price_of_day = promo
        else:
            price_of_day = regular
        if regular is not None and promo is not None:
            price_reduction_pct = round((promo - regular) / regular * 100, 2)
        else:
            price_reduction_pct = None
        entry = {
            "product_id":    p.get("productId"),
            "upc":           p.get("upc"),
            "description":   p.get("description"),
            "brand":         p.get("brand"),
            "category":      p.get("categories", [None])[0],
            "size":          items[0].get("size") if items else None,
            "regular_price": regular,
            "promo_price":   promo,
            "national_regular_price": national_regular,
            "national_promo_price":   national_promo,
            "price_of_day":  price_of_day,
            "price_reduction_%": price_reduction_pct,
            "in_store":      items[0].get("inventory", {}).get("stockLevel") if items else None,
            "ingestion_timestamp": now,
        }
        rows.append(entry)
    return rows

flat = flatten_products(products)
flat = [x for x in flat if isinstance(x, dict)]

schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("upc", StringType(), True),
    StructField("description", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("category", StringType(), True),
    StructField("size", StringType(), True),
    StructField("regular_price", DoubleType(), True),
    StructField("promo_price", DoubleType(), True),
    StructField("national_regular_price", DoubleType(), True),
    StructField("national_promo_price", DoubleType(), True),
    StructField("price_of_day", DoubleType(), True),
    StructField("price_reduction_%", DoubleType(), True),
    StructField("in_store", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
])

df = spark.createDataFrame(flat, schema)

(df.write
   .format("delta")
   .mode("append")
   .option("mergeSchema", "true")
   .saveAsTable("kroger.products"))

print(f"Loaded {df.count()} products")

In [0]:
from pyspark.sql.functions import col, to_date, min, concat_ws, lag
from pyspark.sql import Window

# Read the bronze table and extract date
bronze_df = spark.table("kroger.products").withColumn("date", to_date(col("ingestion_timestamp")))

# Window specification to find earliest ingestion per product and date
window_spec = Window.partitionBy("product_id", "date")

# Mark the earliest ingestion per product/date
silver_df = bronze_df.withColumn(
    "min_ingestion",
    min("ingestion_timestamp").over(window_spec)
).filter(col("ingestion_timestamp") == col("min_ingestion")).drop("min_ingestion")

# Combine description and size for a unique product representation
silver_df = silver_df.withColumn(
    "description_size",
    concat_ws(" | ", col("description"), col("size"))
)

# Compute price change from previous date for each product
delta_window = Window.partitionBy("description_size").orderBy(col("date"))
silver_df = silver_df.withColumn(
    "price_delta",
    col("price_of_day") - lag(col("price_of_day"), 1).over(delta_window)
)

silver_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("kroger.products_silver")
print(f"Silver table row count: {silver_df.count()} rows")


In [0]:
# Gold Table: Duplicate Silver
gold_df = spark.table("kroger.products_silver")
gold_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("kroger.products_gold")
print(f"Gold table row count: {gold_df.count()} rows")

display(spark.table("kroger.products_silver"))
# Silver Table - Deduplicated by Product + Date


In [0]:
display(spark.table("kroger.products"))
# Bronze Table - Raw Products

In [0]:
display(spark.table("kroger.products_silver"))
# Silver Table - Deduplicated by Product + Date

In [0]:
df = spark.table("kroger.products_gold")
on_sale = df.filter(df["price_reduction_%"].isNotNull())
on_sale = on_sale.select("description_size", "price_reduction_%")
on_sale = on_sale.orderBy("price_reduction_%", ascending=True).limit(10)
display(on_sale)
# Top 10 products by price reduction percent (silver table), now sorted so biggest drop (most negative) is first

Databricks visualization. Run in Databricks to view.

In [0]:
%python
display(spark.table("kroger.products_gold"))
# Then click the chart icon → Bar/Scatter/Pie to configure

In [0]:
%sql
SELECT 
  description_size, 
  date,
  `price_reduction_%` 
FROM 
  kroger.products_gold
WHERE 
  `price_reduction_%` IS NOT NULL
  AND date = (SELECT MAX(date) FROM kroger.products_gold)
ORDER BY 
  `price_reduction_%` ASC 
LIMIT 10;

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

gold = spark.table("kroger.products_gold")
latest_date = gold.agg(F.max("date")).collect()[0][0]

on_sale = gold.filter((gold["price_reduction_%"].isNotNull()) & (gold["date"] == latest_date))
on_sale = on_sale.select("description_size", "date", "price_reduction_%")
on_sale = on_sale.orderBy("price_reduction_%", ascending=True).limit(10)

# Overwrite the target table with today's top deals
db_target = "kroger.daily_best_deals"
on_sale.write.format("delta").mode("overwrite").saveAsTable(db_target)

print(f"Saved {on_sale.count()} records to {db_target} for {latest_date}")
display(spark.table(db_target))

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

gold = spark.table("kroger.products_gold")
cheapest_item = gold.filter(gold["price_of_day"].isNotNull()).orderBy("price_of_day", ascending=True).limit(1)
display(cheapest_item)
# Displays the cheapest item among all tracked products, ignoring nulls

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql import functions as F

gold = spark.table("kroger.products_gold")

# Count number of price fluctuations per day (non-null price_delta)
fluctuation_count_df = gold.filter(gold["price_delta"].isNotNull()) \
    .groupBy("date") \
    .agg(F.count("*").alias("num_price_fluctuations"))

display(fluctuation_count_df)
# Shows the number of price fluctuations per day

# Find the day with the most price fluctuations
most_fluctuation_day = fluctuation_count_df.orderBy("num_price_fluctuations", ascending=False).limit(1)

display(most_fluctuation_day)
# Shows the day with the most price fluctuations across all products

In [0]:
from pyspark.sql import functions as F

gold = spark.table("kroger.products_gold")

# Calculate number of items with price drops by date
price_drop_count_df = gold.filter(
    gold["price_delta"].isNotNull() & (gold["price_delta"] < 0)
).groupBy("date") \
 .agg(
     F.count("*").alias("num_price_drops")
 ) \
 .orderBy("date")

display(price_drop_count_df)

In [0]:
from pyspark.sql import functions as F

gold = spark.table("kroger.products_gold")

# Isolate products with price drop on 2026-03-11
dropped_products = gold.filter(
    (gold["date"] == F.lit("2026-03-11")) &
    (gold["price_delta"] < 0)
).select("product_id", "description_size").distinct().limit(68)

# Get their trends over all dates
trends = gold.join(
    dropped_products,
    on=["product_id", "description_size"],
    how="inner"
).orderBy("description_size", "date")

display(trends)